# Collab-YT  - YouTube & TikTok Downloader
Max Quality | Multi-Threading | Google Drive Save

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q gradio yt-dlp
!mkdir -p /content/Collab-YT

In [ ]:
import urllib.request
url = 'https://raw.githubusercontent.com/your-repo/Collab-YT/main/collab_yt.py'
# fallback: write inline
import inspect, textwrap

COLLAB_YT_CODE = '''import os, re, json, shutil, subprocess, threading, zipfilefrom pathlib import Pathfrom datetime import datetimefrom concurrent.futures import ThreadPoolExecutor, as_completedBASE_DIR = Path('/content/Collab-YT')DRIVE_DIR = Path('/content/drive/MyDrive/Collab-YT-Downloads')os.makedirs(DRIVE_DIR, exist_ok=True)QUALITY_MAP = {    'max':     {'label': 'Max (Best Available)',  'fmt': 'bestvideo[height<=2160]+bestaudio/best[height<=2160]'},    '4k':      {'label': '4K (2160p)',             'fmt': 'bestvideo[height<=2160]+bestaudio/best[height<=2160]'},    '1080p':   {'label': 'Full HD (1080p)',         'fmt': 'bestvideo[height<=1080]+bestaudio/best[height<=1080]'},    '720p':    {'label': 'HD (720p)',               'fmt': 'bestvideo[height<=720]+bestaudio/best[height<=720]'},    '480p':    {'label': 'SD (480p)',               'fmt': 'bestvideo[height<=480]+bestaudio/best[height<=480]'},    'audio':   {'label': 'Audio Only',              'fmt': 'bestaudio/best'},}YTDLP_PATH = Nonedef get_ytdlp():    global YTDLP_PATH    if YTDLP_PATH: return YTDLP_PATH    for p in ['yt-dlp', shutil.which('yt-dlp'), str(BASE_DIR / 'yt-dlp')]:        if p and os.path.exists(p) if '/' in (p or '') else shutil.which(p):            YTDLP_PATH = p if '/' in p else shutil.which(p)            return YTDLP_PATH    return Nonedef install_ytdlp():    url = 'https://github.com/yt-dlp/yt-dlp/releases/latest/download/yt-dlp'    path = BASE_DIR / 'yt-dlp'    import urllib.request    urllib.request.urlretrieve(url, str(path))    os.chmod(path, 0o755)    global YTDLP_PATH    YTDLP_PATH = str(path)    return YTDLP_PATHdef check_ffmpeg():    return shutil.which('ffmpeg') or (BASE_DIR / 'ffmpeg').exists()def install_ffmpeg():    url = 'https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz'    archive = BASE_DIR / 'ffmpeg.tar.xz'    import urllib.request    urllib.request.urlretrieve(url, str(archive))    import tarfile    with tarfile.open(archive) as tar:        for member in tar.getmembers():            if member.name.endswith('/ffmpeg') or member.name.endswith('/ffprobe'):                member.name = os.path.basename(member.name)                tar.extract(member, str(BASE_DIR))    archive.unlink()    os.chmod(BASE_DIR / 'ffmpeg', 0o755)    os.chmod(BASE_DIR / 'ffprobe', 0o755)def sanitize(name):    return re.sub(r'[\\\\/*?:\"<>|]', '', name)[:100]def format_size(bytes_):    if bytes_ > 1e9: return f'{bytes_/1e9:.2f} GB'    if bytes_ > 1e6: return f'{bytes_/1e6:.2f} MB'    if bytes_ > 1e3: return f'{bytes_/1e3:.2f} KB'    return f'{bytes_} B'def get_video_info(url):    cmd = get_ytdlp()    if not cmd: return None    r = subprocess.run(        [cmd, '--flat-playlist', '--print', '%(playlist_title)s',         '--print', '%(playlist_count)s',         '--print', '%(id)s | %(title)s | %(duration_string)s | %(view_count)s views',         '--no-download', url],        capture_output=True, text=True, timeout=120    )    lines = r.stdout.strip().split('\n')    if len(lines) >= 3:        return {'title': lines[0], 'count': lines[1], 'videos': lines[2:]}    return Nonedef download_single(info_line, quality_key, output_dir, platform='youtube'):    cmd = get_ytdlp()    if not cmd: return False, 'yt-dlp not found'    parts = info_line.split(' | ')    video_id = parts[0].strip()    title = parts[1].strip() if len(parts) > 1 else f'video_{video_id}'    clean = sanitize(title)    vid_url = f'https://www.youtube.com/watch?v={video_id}' if platform == 'youtube' else video_id    out_path = output_dir / f'{clean}.mp4'    if out_path.exists():        return True, f'Already exists: {clean}'    qf = QUALITY_MAP.get(quality_key, QUALITY_MAP['max'])['fmt']    if quality_key == 'audio':        out = str(output_dir / f'{clean}.%(ext)s')        args = [cmd, '-f', qf, '-o', out, '--no-playlist', '--ignore-errors',                '--extract-audio', '--audio-format', 'mp3', '--audio-quality', '0']    else:        out = str(output_dir / f'{clean}.%(ext)s')        args = [cmd, '-f', qf, '-o', out, '--no-playlist', '--ignore-errors',                '--merge-output-format', 'mp4', '--concurrent-fragments', '8',                '--retries', '10', '--fragment-retries', '10']    if platform == 'tiktok':        args.extend(['--extractor-args', 'tiktok:api_host=web'])    args.append(vid_url)    try:        proc = subprocess.run(args, capture_output=True, text=True, timeout=3600)        if proc.returncode == 0: return True, f'Done: {clean}'        return False, f'Failed: {proc.stderr[:200]}'    except Exception as e:        return False, f'Error: {str(e)}'def download_playlist(url, quality_key, output_dir, workers=4, platform='youtube', progress_cb=None):    cmd = get_ytdlp()    if not cmd: install_ytdlp()    if not check_ffmpeg(): install_ffmpeg()    output_dir = Path(output_dir)    output_dir.mkdir(parents=True, exist_ok=True)    info = get_video_info(url)    if not info:        return {'status': 'error', 'message': 'Failed to fetch video info'}    videos = info['videos']    total = len(videos)    results = []    with ThreadPoolExecutor(max_workers=workers) as executor:        futures = {executor.submit(download_single, v, quality_key, output_dir, platform): v for v in videos}        done_count = 0        for future in as_completed(futures):            ok, msg = future.result()            results.append((ok, msg))            done_count += 1            if progress_cb: progress_cb(done_count, total, msg)    success = sum(1 for ok, _ in results if ok)    return {'status': 'complete', 'total': total, 'success': success, 'failed': total - success, 'output_dir': str(output_dir)}def zip_output(output_dir):    output_dir = Path(output_dir)    zip_name = f'Collab-YT-{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'    zip_path = DRIVE_DIR / zip_name    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:        for f in output_dir.rglob('*'):            if f.is_file(): zf.write(f, f.relative_to(output_dir))    return str(zip_path)'''with open('/content/Collab-YT/collab_yt.py', 'w') as f:    f.write(COLLAB_YT_CODE)from collab_yt import *print('Collab-YT loaded!')

In [ ]:
import gradio as gr
import sys, os, threading, json
sys.path.insert(0, '/content/Collab-YT')
from collab_yt import *

OUTPUT_DIR = Path('/content/Collab-YT/downloads')

def fetch_info(url):
    if not url:
        return '⚠️ Please enter a URL'
    info = get_video_info(url)
    if info:
        lines = info['videos'][:10]
        text = f"""[36m{'='*60}[0m
[33m[1mChannel:[0m {info['title']}
[33m[1mVideos:[0m {info['count']}
[36m{'='*60}[0m
"""
        for l in lines:
            p = l.split(' | ')
            if len(p) >= 3:
                text += f"[34m{p[0].strip()}[0m | [37m{p[1].strip()[:60]}[0m | [32m{p[2].strip()}[0m\n"
        return text
    return '❌ Failed to fetch info. Check URL.'

def download_task(url, quality, workers, platform, progress=gr.Progress()):
    progress(0, desc='Starting...')
    
    if not get_ytdlp():
        yield 'Installing yt-dlp...'
        install_ytdlp()
    if not check_ffmpeg():
        yield 'Installing FFmpeg...'
        install_ffmpeg()
    
    sub_dir = 'youtube' if platform == 'YouTube' else 'tiktok'
    out_dir = OUTPUT_DIR / sub_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    
    q_key = quality.lower().split(' ')[0]
    if q_key == 'max': q_key = 'max'
    
    yield f'🚀 Downloading {url}...'
    
    result = download_playlist(
        url, q_key, out_dir,
        workers=int(workers),
        platform=platform.lower(),
        progress_cb=lambda d, t, m: progress(d/t, desc=f'[{d}/{t}] {m[:50]}')
    )
    
    if result['status'] == 'error':
        yield f'❌ {result["message"]}'
        return
    
    yield f"""[32m[1m{'='*50}[0m
[32m[1m✅ Complete![0m
[33mSuccess:[0m {result['success']}/{result['total']}
[31mFailed:[0m {result['failed']}
[36mOutput:[0m {result['output_dir']}
[32m[1m{'='*50}[0m"""
    
    progress(1.0, desc='Done!')

def save_to_drive():
    if not any(OUTPUT_DIR.rglob('*')):
        return '⚠️ No downloads to save.'
    zip_path = zip_output(OUTPUT_DIR)
    return f'✅ Saved to Drive:\n{zip_path}'

with gr.Blocks(title='Collab-YT', theme='Soft', css='''
    .gradio-container { max-width: 900px !important; margin: auto !important; }
    h1 { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
         -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    footer { display: none !important; }
''') as demo:
    gr.Markdown('#  Collab-YT', description='YouTube + TikTok Downloader')
    gr.Markdown('Max Quality | Multi-Threading | Google Drive')
    
    with gr.Row():
        with gr.Column(scale=2):
            url_input = gr.Textbox(label='URL', placeholder='YouTube/TikTok URL...', lines=1)
            with gr.Row():
                quality_dd = gr.Dropdown(
                    choices=['Max (Best)', '4K', '1080p', '720p', '480p', 'Audio Only'],
                    value='Max (Best)', label='Quality'
                )
                workers_slider = gr.Slider(1, 16, value=4, step=1, label='Threads')
            with gr.Row():
                platform_dd = gr.Radio(['YouTube', 'TikTok'], value='YouTube', label='Platform')
            with gr.Row():
                fetch_btn = gr.Button(' Preview', variant='secondary', size='sm')
                dl_btn = gr.Button(' Download', variant='primary', size='lg')
        
        with gr.Column(scale=1):
            drive_btn = gr.Button(' Save to Google Drive', variant='secondary')
            drive_out = gr.Textbox(label='Drive Status', lines=2, interactive=False)
    
    info_out = gr.Textbox(label='Video Info', lines=8, interactive=False)
    log_out = gr.Textbox(label='Log', lines=8, interactive=False)
    
    fetch_btn.click(fn=fetch_info, inputs=url_input, outputs=info_out)
    dl_btn.click(
        fn=download_task,
        inputs=[url_input, quality_dd, workers_slider, platform_dd],
        outputs=log_out
    )
    drive_btn.click(fn=save_to_drive, outputs=drive_out)
    
    gr.Markdown('---')
    gr.Markdown('### Quick Tips')
    gr.Markdown('- Use **Preview** to see videos before downloading')
    gr.Markdown('- Max quality downloads best available up to 4K')
    gr.Markdown('- Files auto-categorized in youtube/ & tiktok/ folders')
    gr.Markdown('- Click **Save to Google Drive** to backup your downloads')

demo.queue()
demo.launch(share=True, debug=False, show_api=False)